# Statement Forecast: empirical-Bayes dynamic panel

Independent research lab. Not affiliated with any employer or financial institution. Not investment advice.

Liu, L., Moon, H. R., & Schorfheide, F., *Forecasting with Dynamic Panel Data Models*,
NBER Working Paper 25102 (2018); *Econometrica* 88(1), 171–201 (2020).
[Paper](https://www.nber.org/papers/w25102). Integrated likelihood: eq. 24;
parametric Tweedie posterior mean: eqs. 17, 20, 30.

The fixed specification is loaded below from `test_design.json`, including the exact pass bar.
Three quarterly lines, targets 2022Q1–2026Q2, origin-ranked top 50, T=12 rolling equations,
lagged-asset scaling, centered seasonal dummies, Gaussian CRE QMLE and Tweedie correction.
All four methods use the shared Y-9C evaluation cases. No tuning, trimming or outlier removal.

The real-data run requires the specification to match its committed HEAD version. This notebook
is distributed unexecuted; run once on real data and publish the result whatever the verdict.

In [ ]:
from pathlib import Path
import sys, os, json, hashlib, subprocess
from datetime import datetime, timezone
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'tracks/y9c-panel/core_items.yaml').exists())
for track in ['tracks/y9c-panel', 'tracks/statement-forecast']:
    sys.path.insert(0, str(ROOT / track))
from y9c.panel import PROCESSED
from statement_forecast.paths import DESIGN_PATH, RESULTS, load_design
from statement_forecast.evaluate import run_all, write_outputs

run_start = datetime.now(timezone.utc).isoformat()
def git(*args):
    return subprocess.check_output(['git', *args], cwd=ROOT)
relative_design = str(DESIGN_PATH.relative_to(ROOT))
try:
    committed = git('show', 'HEAD:' + relative_design)
except subprocess.CalledProcessError as exc:
    raise RuntimeError('Pre-registration must be committed separately before the real-data run') from exc
digest = hashlib.sha256(DESIGN_PATH.read_bytes()).hexdigest()
if digest != hashlib.sha256(committed).hexdigest():
    raise ValueError('test_design.json differs from the committed HEAD pre-registration')
print('Verified test_design.json SHA-256:', digest)
prereg_commit = git('log', '--format=%H', '-n1', '--', relative_design).decode().strip()
if not prereg_commit:
    raise ValueError('Missing pre-registration commit')
run_meta = dict(run_start_time_utc=run_start, git_head=git('rev-parse', 'HEAD').decode().strip(),
                prereg_commit=prereg_commit,
                prereg_commit_time=git('show', '-s', '--format=%cI', prereg_commit).decode().strip(),
                test_design_sha256=digest)
design = load_design()
display(Markdown(design['research_only']))
display(Markdown('**Question:** ' + design['question']))
display(pd.DataFrame(design['lines']))
display(Markdown('**Pass bar:** ' + design['pass_bar']['rule']))
print(json.dumps({key: design[key] for key in ['test_window', 'universe', 'evaluation_cases', 'method']}, indent=2))

panel_path = Path(os.environ['SF_PANEL_PATH']) if 'SF_PANEL_PATH' in os.environ else PROCESSED / 'panel_wide.parquet'
results_dir = Path(os.environ.get('SF_RESULTS_DIR', str(RESULTS)))
# A panel override takes its sibling vintage by default, avoiding the real cache in smoke tests.
vintage_path = Path(os.environ.get('SF_VINTAGE_PATH', str(panel_path.parent / 'vintage.json')))
panel = pd.read_parquet(panel_path)
print('JPMorgan RSSD 1039502; 2024+ quarterly flows, thousands USD')
print(panel.loc[panel.rssd_id.eq(1039502) & panel.report_date.dt.year.ge(2024),
                ['report_date', *[line['column'] for line in design['lines']]]]
      .sort_values('report_date').to_string(index=False))

## Fixed out-of-sample evaluation

Estimation uses the origin-selected cross-section, independent of whether a next-quarter target exists.
Incomplete ratio windows receive the naive fallback on evaluation cases, with counts reported.
Numerical failures raise. MAE/RMSE are thousands USD, equally weighted by BHC-quarter.
Positive DM t favors EB; two-sided p uses target-quarter CR1 clusters and Student t(G−1).
Per-year results and parameter summaries are **DESCRIPTIVE ONLY**; there is no multiplicity adjustment.

In [ ]:
results = run_all(panel, design)
print(results['headline'])
error_format = {'mae': '{:,.0f}', 'rmse': '{:,.0f}'}
for line in design['lines']:
    key = line['key']
    display(Markdown('### ' + line['label']))
    display(results['overall_errors'].query('line == @key').style.format(error_format))
    dm = results['dm_tests'].query('line == @key')
    display(dm.loc[dm.scope.eq('full')].style.format({'t_stat':'{:.3f}', 'p_value':'{:.4f}'}))
    print(json.dumps(results['verdicts'][key], indent=2))
    print('PER-YEAR ERRORS AND DM: DESCRIPTIVE ONLY')
    display(results['by_year_errors'].query('line == @key').style.format(error_format))
    display(dm.loc[dm.scope.ne('full')].style.format({'t_stat':'{:.3f}', 'p_value':'{:.4f}'}))
    print('EB fallbacks:', results['audits'][key]['eb_fallbacks'])
print('EB PARAMETER SUMMARY: DESCRIPTIVE ONLY')
display(results['eb_parameters'].groupby('line')[['rho', 'shrinkage', 'omega2']].agg(['min', 'median', 'max']))
write_outputs(results, results_dir, vintage_path, run_meta)

## Aggregate figures

Navy palette; MAE in whole thousands USD. Parameter paths are descriptive only.
No per-bank forecast rows are exported.

In [ ]:
COLORS = ['#0a2540', '#0a4080', '#5a7a9a', '#8daabe']
LABELS = {'eb_panel':'EB panel', 'naive':'Naive', 'seasonal_naive':'Seasonal naive', 'pooled_ar':'Pooled panel AR'}
plt.rcParams.update({'axes.prop_cycle': plt.cycler(color=COLORS), 'figure.dpi':130,
                     'axes.labelcolor':'#0a2540', 'text.color':'#0a2540',
                     'xtick.color':'#0a2540', 'ytick.color':'#0a2540', 'axes.edgecolor':'#5a7a9a',
                     'grid.color':'#8daabe', 'legend.fancybox':False,
                     'figure.facecolor':'white', 'axes.facecolor':'white'})
figures = results_dir / 'figures'
figures.mkdir(parents=True, exist_ok=True)
def save(fig, filename):
    fig.tight_layout()
    fig.savefig(figures / filename, dpi=130, bbox_inches='tight')
    plt.show()
    plt.close(fig)

mae = results['overall_errors'].pivot(index='line', columns='method', values='mae')
mae = mae.reindex(index=[line['key'] for line in design['lines']], columns=list(LABELS))
mae.index = [line['label'] for line in design['lines']]
fig, ax = plt.subplots(figsize=(10, 5))
mae.rename(columns=LABELS).plot.bar(ax=ax, color=COLORS, rot=0)
ax.set(xlabel='Statement line', ylabel='MAE (thousands USD)', title='One-quarter-ahead forecast MAE')
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, pos: f'{value:,.0f}'))
ax.legend(title='Method', loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)
save(fig, 'mae_by_line.png')

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for line in design['lines']:
    params = results['eb_parameters'].loc[results['eb_parameters'].line.eq(line['key'])]
    dates = pd.PeriodIndex(params.origin, freq='Q').to_timestamp(how='end').normalize()
    axes[0].plot(dates, params.rho, label=line['label'])
    axes[1].plot(dates, params.shrinkage, label=line['label'])
axes[0].set(ylabel='Estimated rho', title='Rolling EB parameters — descriptive only')
axes[1].set(xlabel='Origin quarter', ylabel='Shrinkage toward prior')
axes[0].legend()
save(fig, 'eb_parameters.png')

## Interpretation and limitations

The headline reports all three pre-registered verdicts, including failures. MAE must strictly improve
on naive and the full-sample absolute-error DM p must be ≤ 0.05; undefined p fails. Other comparisons
and yearly tables are descriptive only. QMLE refits at each origin and posterior means use the
Gaussian CRE score. Seasonal dummies and quarterly observations, the top-50 cross-section, no macro
covariates, and no outlier elimination distinguish this application from the paper.

This is a report-quarter backtest on currently downloaded filings: filing delays and revisions are
not modeled. RSSD histories have no pro-forma merger adjustments, acquisition jumps remain, and
missing calendar predecessors are not bridged. Parent/subsidiary filers are not consolidated.
Dollar losses emphasize large institutions. Research only; not investment advice.